In [1]:
import requests
import pandas as pd
import os

In [2]:
fire_event_name = "YORK_2024-08-03_5"
job_id = '573421be-412c-403d-9d72-b591de886c43'
test = requests.get(f"https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/result/analyze_fire_severity/{fire_event_name}/{job_id}")
print(test.json())

{'fire_event_name': 'YORK_2024-08-03_5', 'status': 'pending', 'job_id': '573421be-412c-403d-9d72-b591de886c43'}


In [3]:
fires = pd.read_csv('fire_processing_jobs.csv')

# Collect status for each row
statuses = []

for idx, row in fires.iterrows():
    # Skip rows with missing fire_event_name or job_id
    if pd.isna(row['fire_event_name']) or pd.isna(row['job_id']):
        statuses.append(999)
        continue
        
    if row['status'] == 'success':
        fire_event_name = row['fire_event_name']
        job_id = row['job_id']

        request = requests.get(f"https://fire-recovery-backend-dev-113009620257.us-central1.run.app/fire-recovery/result/analyze_fire_severity/{fire_event_name}/{job_id}")
        
        status = request.json().get('status')
        print(f"Job {fire_event_name}: {status}")
        
    else:
        status = 999
    
    statuses.append(status)

# Add status column to dataframe
fires['job_status'] = statuses

# Save updated dataframe
fires.to_csv('fire_processing_jobs.csv', index=False)

print(f"\nUpdated fire_processing_jobs.csv with job_status column")
fires.head()

Job COFFEE POT_date2024-08-03_range5_modealarm: complete
Job COFFEE POT_date2024-08-03_range10_modealarm: complete
Job COFFEE POT_date2024-08-03_range15_modealarm: complete
Job COFFEE POT_date2024-08-03_range21_modealarm: complete
Job COFFEE POT_date2024-08-03_range30_modealarm: complete
Job COFFEE POT_date2024-08-03_range45_modealarm: complete
Job COFFEE POT_date2024-08-03_range60_modealarm: complete
Job COFFEE POT_date2024-08-03_range90_modealarm: complete
Job COFFEE POT_date2024-12-16_range5_modecont: complete
Job COFFEE POT_date2024-12-16_range10_modecont: complete
Job COFFEE POT_date2024-12-16_range15_modecont: complete
Job COFFEE POT_date2024-12-16_range21_modecont: complete
Job COFFEE POT_date2024-12-16_range30_modecont: complete
Job COFFEE POT_date2024-12-16_range45_modecont: complete
Job COFFEE POT_date2024-12-16_range60_modecont: complete
Job COFFEE POT_date2024-12-16_range90_modecont: complete
Job SENTINEL_date2024-07-14_range5_modealarm: complete
Job SENTINEL_date2024-07-14

,fire_event_name,job_id,fire_name,date_mode,post_fire_days,status,job_status
0,COFFEE POT_date2024-08-03_range5_modealarm,674b9f1e-0dd5-43b6-9717-998a5c8b10e9,COFFEE POT,alarm,5,success,complete
1,COFFEE POT_date2024-08-03_range10_modealarm,36dce962-a982-4e60-af53-eb061e0c6428,COFFEE POT,alarm,10,success,complete
2,COFFEE POT_date2024-08-03_range15_modealarm,7baa9569-e731-4d08-8869-7d2fcdbac731,COFFEE POT,alarm,15,success,complete
3,COFFEE POT_date2024-08-03_range21_modealarm,9b02f9a0-3373-4a8c-8743-1c58ab2f8b5c,COFFEE POT,alarm,21,success,complete
4,COFFEE POT_date2024-08-03_range30_modealarm,eadd650b-94ba-4088-8b3f-9a3666ff4e1b,COFFEE POT,alarm,30,success,complete


In [ ]:
urls = request.json().get('coarse_severity_cog_urls')
os.makedirs('data/raw', exist_ok=True)

# Download rbr and dnbr files
for metric in ['rbr', 'dnbr']:
    response = requests.get(urls[metric])
    with open(f"data/raw/{fire_event_name}_{metric}.tif", 'wb') as f:
        f.write(response.content)

print(f"Job {fire_event_name} downloaded.")